In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("file_date", "2024-12-16", "File Date")
file_date = dbutils.widgets.get("file_date")

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit
from delta.tables import DeltaTable

In [0]:
production_country_df = (
    spark.table("databricks_ws.bronze.production_countries")
    .filter(col("file_date") == file_date)
    )

In [0]:
production_country_selected_df = (
    production_country_df.select(
        col("movieId").cast("integer"),
        col("countryId").cast("integer"),
        col("file_date")
        )
    )

In [0]:
production_countries_final_df = (
    add_ingestion_date(production_country_selected_df)
    .withColumnRenamed("movieId", "movie_id")
    .withColumnRenamed("countryId", "country_id")
    .withColumn("environment", lit("Production"))
)

In [0]:
table_name = "databricks_ws.silver.production_countries"
if spark.catalog.tableExists(table_name):

    deltaTable = DeltaTable.forName(spark, table_name)

    deltaTable.alias("tgt") \
        .merge(
            production_countries_final_df.alias("src"),
            "tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    production_countries_final_df.write \
        .format("delta") \
        .saveAsTable(table_name)